# 💪 Gym Tracker — Real-Time Bicep Curl Counter

A real-time **computer vision-based gym tracker** that uses **MediaPipe Pose** and **OpenCV** to detect body landmarks, calculate joint angles, and automatically count bicep curls for both the left and right arms.

The system processes the webcam feed in real time and provides visual feedback about:

* 🦾 Left and right arm pose
* 🔢 Repetition count
* 📐 Elbow joint angle
* ⬆️⬇️ Current movement stage
* 🟢 Live workout status

## 🚀 Features

* Real-time human pose detection
* 33-point body landmark detection using MediaPipe
* Elbow angle calculation using three-point geometry
* Independent left and right arm repetition tracking
* Automatic `UP` / `DOWN` movement detection
* Real-time visual dashboard
* Live webcam processing using OpenCV

## 🛠️ Technologies Used

| Technology | Purpose                                      |
| ---------- | -------------------------------------------- |
| Python     | Core programming language                    |
| OpenCV     | Webcam capture and video processing          |
| MediaPipe  | Human pose estimation                        |
| NumPy      | Numerical calculations and angle computation |

## 🎯 Objective

The goal of this project is to build a simple **AI-powered fitness tracking system** that can understand human body movements through a webcam without requiring specialized hardware.


In [1]:
import cv2
import mediapipe as mp
import numpy as np
mp_drawing = mp.solutions.drawing_utils
mp_pose = mp.solutions.pose

In [2]:
#Video Feed
cap = cv2.VideoCapture(0)
while cap.isOpened():
    ret, frame = cap.read()
    cv2.imshow("Mediapipe feed", frame)
    if cv2.waitKey(10) & 0xFF == ord('q'):
        break
cap.release()
cv2.destroyAllWindows()

## 1. 📷 Capturing the Video Feed

The first step is to access the computer's webcam using OpenCV.

Each frame from the webcam is continuously captured and displayed. This provides the real-time video stream that will later be processed by the pose estimation model.


In [3]:
cap = cv2.VideoCapture(0)
#Setting MediaPipe Instance
with mp_pose.Pose(min_detection_confidence = 0.5, min_tracking_confidence = 0.5) as pose:
    while cap.isOpened():
        ret, frame = cap.read()
        
        #Recolor Image to RGB
        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        image.flags.writeable = False
        
        #Make detection
        results = pose.process(image)
        
        #Recolor Image to BGR
        image.flags.writeable = True
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

        #Drawing landmarks
        mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_pose.POSE_CONNECTIONS,
                                 mp_drawing.DrawingSpec(color = (245,117,66),thickness = 2,circle_radius = 2),
                                 mp_drawing.DrawingSpec(color = (245,66,230),thickness = 2,circle_radius = 2),
                                 )
        
        cv2.imshow("Mediapipe feed", image)
        if cv2.waitKey(10) & 0xFF == ord('q'):
            break
cap.release()
cv2.destroyAllWindows()

## 2. 🧍 Detecting Body Landmarks

MediaPipe Pose is used to detect human body landmarks from each video frame.

The model identifies **33 landmarks** across the human body, including the:

* Shoulders
* Elbows
* Wrists
* Hips
* Knees
* Ankles
* Head

For the bicep curl tracker, the most important landmarks are:

**Shoulder → Elbow → Wrist**

These three points allow the system to determine the angle of the elbow and understand the position of the arm during a curl.


In [4]:
cap = cv2.VideoCapture(0)
#Setting MediaPipe Instance
with mp_pose.Pose(min_detection_confidence = 0.5, min_tracking_confidence = 0.5) as pose:
    while cap.isOpened():
        ret, frame = cap.read()
        
        #Recolor Image to RGB
        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        image.flags.writeable = False
        
        #Make detection
        results = pose.process(image)
        
        #Recolor Image to BGR
        image.flags.writeable = True
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

        try:
            landmarks = results.pose_landmarks.landmark
            print(landmarks)
        except:
            pass
        mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_pose.POSE_CONNECTIONS,
                                 mp_drawing.DrawingSpec(color = (245,117,66),thickness = 2,circle_radius = 2),
                                 mp_drawing.DrawingSpec(color = (245,66,230),thickness = 2,circle_radius = 2),
                                 )
        
        cv2.imshow("Mediapipe feed", image)
        if cv2.waitKey(10) & 0xFF == ord('q'):
            break
cap.release()
cv2.destroyAllWindows()

[x: 0.3879539966583252
y: 0.4395354092121124
z: -1.0147225856781006
visibility: 0.9998119473457336
, x: 0.41091108322143555
y: 0.38066741824150085
z: -0.9532820582389832
visibility: 0.9996976852416992
, x: 0.42753058671951294
y: 0.3807636499404907
z: -0.9534655809402466
visibility: 0.9997770190238953
, x: 0.4440085291862488
y: 0.38018715381622314
z: -0.9538971781730652
visibility: 0.9996931552886963
, x: 0.3507988452911377
y: 0.3791106939315796
z: -0.9612761735916138
visibility: 0.9996579885482788
, x: 0.33109575510025024
y: 0.3791459798812866
z: -0.9604557156562805
visibility: 0.9997476935386658
, x: 0.3128572106361389
y: 0.3792998492717743
z: -0.9609829783439636
visibility: 0.9996722936630249
, x: 0.47032660245895386
y: 0.40997886657714844
z: -0.5002793073654175
visibility: 0.9997894167900085
, x: 0.28910887241363525
y: 0.41387224197387695
z: -0.5143723487854004
visibility: 0.9998151659965515
, x: 0.42350929975509644
y: 0.5078670978546143
z: -0.8390692472457886
visibility: 0.99985861

In [5]:
len(landmarks)


33

In [6]:
landmarks[mp_pose.PoseLandmark.LEFT_KNEE.value]

x: 0.5820657014846802
y: 2.2354164123535156
z: -0.24916839599609375
visibility: 0.0007255729287862778

## 3. 📐 Calculating the Elbow Angle

To determine whether the arm is extended or curled, the angle at the elbow is calculated using the detected shoulder, elbow, and wrist coordinates.

The angle is calculated using the two vectors:

**Elbow → Shoulder**

and

**Elbow → Wrist**

The resulting angle is converted from radians to degrees.

### Angle interpretation

| Elbow Angle | Arm Position          |
| ----------: | --------------------- |
|    `> 160°` | Arm extended — `DOWN` |
|     `< 30°` | Arm curled — `UP`     |

This angle-based approach allows the system to detect the two major stages of a bicep curl.


In [7]:
def calculate_angle(a,b,c):
    a = np.array(a)
    b = np.array(b)
    c = np.array(c)

    radians = np.arctan2(c[1]-b[1], c[0]-b[0]) - np.arctan2(a[1]-b[1], a[0]-b[0])
    angle = np.abs(radians * 180.0/np.pi)
    if angle>180:
        return 360 - angle
    return angle

In [8]:
shoulder = [landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].x, landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].y]
elbow = [landmarks[mp_pose.PoseLandmark.LEFT_ELBOW.value].x, landmarks[mp_pose.PoseLandmark.LEFT_ELBOW.value].y]
wrist = [landmarks[mp_pose.PoseLandmark.LEFT_WRIST.value].x, landmarks[mp_pose.PoseLandmark.LEFT_WRIST.value].y]


In [9]:
shoulder, elbow, wrist

([0.6419202089309692, 0.7213422656059265],
 [0.7845317721366882, 0.9876880049705505],
 [0.8605645298957825, 1.268219232559204])

In [10]:
calculate_angle(shoulder, elbow, wrist)

166.9983602868958

## 4. 🔢 Real-Time Bicep Curl Tracking

Once the body landmarks and elbow angles are available, the system can track repetitions.

A repetition is counted using a simple **state-based approach**:

1. The arm first reaches the `DOWN` position.
2. The elbow angle decreases below `30°`.
3. The stage changes from `DOWN` → `UP`.
4. The repetition counter increases by one.
5. The arm must return to the `DOWN` position before another repetition can be counted.

The left and right arms are tracked independently, allowing separate repetition counts for each arm.


In [16]:
import cv2
import mediapipe as mp
import numpy as np

cap = cv2.VideoCapture(0)

# Camera resolution
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)

l_counter = 0
l_stage = "DOWN"

r_counter = 0
r_stage = "DOWN"

mp_pose = mp.solutions.pose
mp_drawing = mp.solutions.drawing_utils


def draw_panel(image, x1, y1, x2, y2, title, reps, stage, angle):
    """
    Draws a clean fitness dashboard panel.
    """

    # Semi-transparent panel
    overlay = image.copy()

    cv2.rectangle(
        overlay,
        (x1, y1),
        (x2, y2),
        (25, 25, 25),
        -1
    )

    # Transparency
    image[:] = cv2.addWeighted(overlay, 0.75, image, 0.25, 0)

    # Border
    cv2.rectangle(
        image,
        (x1, y1),
        (x2, y2),
        (255, 255, 255),
        2
    )

    # Title
    cv2.putText(
        image,
        title,
        (x1 + 20, y1 + 40),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (255, 255, 255),
        2,
        cv2.LINE_AA
    )

    # REP label
    cv2.putText(
        image,
        "REPS",
        (x1 + 20, y1 + 85),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.55,
        (180, 180, 180),
        1,
        cv2.LINE_AA
    )

    # Rep count
    cv2.putText(
        image,
        str(reps),
        (x1 + 20, y1 + 150),
        cv2.FONT_HERSHEY_DUPLEX,
        2.2,
        (255, 255, 255),
        3,
        cv2.LINE_AA
    )

    # Stage
    cv2.putText(
        image,
        "STAGE",
        (x1 + 150, y1 + 85),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.55,
        (180, 180, 180),
        1,
        cv2.LINE_AA
    )

    # Stage color
    if stage == "UP":
        stage_color = (0, 255, 0)
    else:
        stage_color = (0, 165, 255)

    cv2.putText(
        image,
        stage,
        (x1 + 150, y1 + 125),
        cv2.FONT_HERSHEY_DUPLEX,
        0.9,
        stage_color,
        2,
        cv2.LINE_AA
    )

    # Angle
    cv2.putText(
        image,
        f"ANGLE  {int(angle)}",
        (x1 + 150, y1 + 165),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.55,
        (220, 220, 220),
        1,
        cv2.LINE_AA
    )


with mp_pose.Pose(
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5
) as pose:

    while cap.isOpened():

        ret, frame = cap.read()

        if not ret:
            break

        # Flip camera horizontally for a mirror-like view
        frame = cv2.flip(frame, 1)

        # Get actual frame dimensions
        h, w, _ = frame.shape

        # -----------------------------------
        # MEDIAPIPE
        # -----------------------------------

        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        image.flags.writeable = False

        results = pose.process(image)

        image.flags.writeable = True
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

        l_angle = 0
        r_angle = 0

        try:

            landmarks = results.pose_landmarks.landmark

            # -----------------------------
            # LEFT ARM
            # -----------------------------

            l_shoulder = [
                landmarks[
                    mp_pose.PoseLandmark.LEFT_SHOULDER.value
                ].x,
                landmarks[
                    mp_pose.PoseLandmark.LEFT_SHOULDER.value
                ].y
            ]

            l_elbow = [
                landmarks[
                    mp_pose.PoseLandmark.LEFT_ELBOW.value
                ].x,
                landmarks[
                    mp_pose.PoseLandmark.LEFT_ELBOW.value
                ].y
            ]

            l_wrist = [
                landmarks[
                    mp_pose.PoseLandmark.LEFT_WRIST.value
                ].x,
                landmarks[
                    mp_pose.PoseLandmark.LEFT_WRIST.value
                ].y
            ]

            l_angle = calculate_angle(
                l_shoulder,
                l_elbow,
                l_wrist
            )

            # -----------------------------
            # RIGHT ARM
            # -----------------------------

            r_shoulder = [
                landmarks[
                    mp_pose.PoseLandmark.RIGHT_SHOULDER.value
                ].x,
                landmarks[
                    mp_pose.PoseLandmark.RIGHT_SHOULDER.value
                ].y
            ]

            r_elbow = [
                landmarks[
                    mp_pose.PoseLandmark.RIGHT_ELBOW.value
                ].x,
                landmarks[
                    mp_pose.PoseLandmark.RIGHT_ELBOW.value
                ].y
            ]

            r_wrist = [
                landmarks[
                    mp_pose.PoseLandmark.RIGHT_WRIST.value
                ].x,
                landmarks[
                    mp_pose.PoseLandmark.RIGHT_WRIST.value
                ].y
            ]

            r_angle = calculate_angle(
                r_shoulder,
                r_elbow,
                r_wrist
            )

            # -----------------------------------
            # REP COUNTING
            # -----------------------------------

            # LEFT
            if l_angle > 160:
                l_stage = "DOWN"

            if l_angle < 30 and l_stage == "DOWN":
                l_stage = "UP"
                l_counter += 1

            # RIGHT
            if r_angle > 160:
                r_stage = "DOWN"

            if r_angle < 30 and r_stage == "DOWN":
                r_stage = "UP"
                r_counter += 1

            # -----------------------------------
            # ANGLE DISPLAY
            # -----------------------------------

            l_elbow_pixel = (
                int(l_elbow[0] * w),
                int(l_elbow[1] * h)
            )

            r_elbow_pixel = (
                int(r_elbow[0] * w),
                int(r_elbow[1] * h)
            )

            # Angle background
            cv2.circle(
                image,
                l_elbow_pixel,
                28,
                (25, 25, 25),
                -1
            )

            cv2.circle(
                image,
                r_elbow_pixel,
                28,
                (25, 25, 25),
                -1
            )

            # Angle text
            cv2.putText(
                image,
                f"{int(l_angle)}",
                (
                    l_elbow_pixel[0] - 18,
                    l_elbow_pixel[1] + 7
                ),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.55,
                (255, 255, 255),
                2,
                cv2.LINE_AA
            )

            cv2.putText(
                image,
                f"{int(r_angle)}",
                (
                    r_elbow_pixel[0] - 18,
                    r_elbow_pixel[1] + 7
                ),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.55,
                (255, 255, 255),
                2,
                cv2.LINE_AA
            )

        except Exception:
            pass

        # -----------------------------------
        # POSE DRAWING
        # -----------------------------------

        if results.pose_landmarks:

            mp_drawing.draw_landmarks(
                image,
                results.pose_landmarks,
                mp_pose.POSE_CONNECTIONS,

                mp_drawing.DrawingSpec(
                    color=(245, 117, 66),
                    thickness=3,
                    circle_radius=4
                ),

                mp_drawing.DrawingSpec(
                    color=(245, 66, 230),
                    thickness=3,
                    circle_radius=3
                )
            )

        # -----------------------------------
        # TOP HEADER
        # -----------------------------------

        overlay = image.copy()

        cv2.rectangle(
            overlay,
            (0, 0),
            (w, 65),
            (15, 15, 15),
            -1
        )

        image[:] = cv2.addWeighted(
            overlay,
            0.8,
            image,
            0.2,
            0
        )

        cv2.putText(
            image,
            "BICEP CURL TRACKER",
            (25, 42),
            cv2.FONT_HERSHEY_DUPLEX,
            1.1,
            (255, 255, 255),
            2,
            cv2.LINE_AA
        )

        # Live indicator
        cv2.circle(
            image,
            (w - 130, 30),
            10,
            (0, 255, 0),
            -1
        )

        cv2.putText(
            image,
            "LIVE",
            (w - 110, 37),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.65,
            (255, 255, 255),
            2,
            cv2.LINE_AA
        )

        # -----------------------------------
        # DASHBOARD PANELS
        # -----------------------------------

        draw_panel(
            image,
            20,
            85,
            400,
            270,
            "LEFT ARM",
            l_counter,
            l_stage,
            l_angle
        )

        draw_panel(
            image,
            w - 400,
            85,
            w - 20,
            270,
            "RIGHT ARM",
            r_counter,
            r_stage,
            r_angle
        )

        # -----------------------------------
        # INSTRUCTIONS
        # -----------------------------------

        cv2.putText(
            image,
            "Press Q to quit",
            (25, h - 25),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.55,
            (200, 200, 200),
            1,
            cv2.LINE_AA
        )

        # -----------------------------------
        # DISPLAY
        # -----------------------------------

        cv2.imshow(
            "Bicep Curl Tracker",
            image
        )

        if cv2.waitKey(10) & 0xFF == ord('q'):
            break


cap.release()
cv2.destroyAllWindows()

## 5. 📊 Output

The final application provides a real-time fitness dashboard containing:

* **Left arm repetition count**
* **Right arm repetition count**
* **Current stage (`UP` / `DOWN`)**
* **Elbow angle**
* **Live pose landmarks**
* **Real-time webcam feed**

The system can therefore act as a basic webcam-based **rep counter and exercise tracking system** without requiring wearable sensors.

## 🧠 How It Works

```text
Webcam
   ↓
OpenCV Video Capture
   ↓
MediaPipe Pose Estimation
   ↓
33 Body Landmarks
   ↓
Extract Shoulder / Elbow / Wrist
   ↓
Calculate Elbow Angle
   ↓
Determine UP / DOWN Stage
   ↓
Count Repetitions
   ↓
Display Real-Time Fitness Dashboard
```

## 🔮 Future Improvements

The current implementation can be extended into a more complete AI fitness assistant by adding:

* Exercise classification
* Squat and push-up tracking
* Automatic form correction
* Rep quality scoring
* Workout history and statistics
* Calorie estimation
* Voice feedback
* Multiple exercise support
* User-specific exercise targets
* Web or mobile dashboard

## 📌 Project Highlights

This project demonstrates practical experience with:

**Computer Vision · Pose Estimation · MediaPipe · OpenCV · NumPy · Real-Time Video Processing · Geometric Angle Calculation · State-Based Rep Counting**
